# Инференс SegFormer и построение 40D векторов (Фаза 4)

Этот ноутбук берёт нарезанные тайлы (800x800) и веса обученной модели SegFormer.
На выходе получается `building_features.csv` с 15-мерным вектором признаков для каждого широкого плана фасада.

In [1]:
!pip install -q transformers

In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from PIL import Image
from scipy.stats import skew as scipy_skew


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем: {device}")

Используем: cuda


## 1. Пути и Настройки

In [3]:
# ПУТИ В KAGGLE
TILES_DIR = '/kaggle/input/datasets/neuromant/tiles-800x800-300-imgs/tiles_800x800_inference-300' 
MODEL_PATH = '/kaggle/input/models/neuromant/segformer-b2-final-fit-408/pytorch/default/1/final_fit_408' 

# EPSILON = 0.01 # Порог для prevalence (1% пикселей)

CONFIDENCE_THRESHOLD = 0.25  # Порог softmax-вероятности для "детектирования" пикселя.
                              # argmax (старый способ) эквивалентен порогу ~0.50.
                              # При 0.25 ловим слабые, но реальные повреждения.
EPSILON = 0.05               # Порог для prevalence — минимальный ratio тайла,
                              # чтобы считаться "повреждённым". Немного повысили
                              # чтобы компенсировать более чувствительный детектор.

CLASS_MAPPING = {
    0: "background",
    1: "coating_deterioration",
    2: "cracks",
    3: "masonry_degradation",
    4: "moisture_bio_damage",
    5: "vandalism",
}

CLASSES_TO_TRACK = [
    "coating_deterioration", 
    "cracks", 
    "masonry_degradation", 
    "moisture_bio_damage", 
    "vandalism"
]


## 2. Загрузка модели

In [4]:
print("Загрузка процессора и модели...")
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()
print("Готово!")

Загрузка процессора и модели...


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Готово!


## 3. Инференс по тайлам

In [5]:
tile_files = glob(os.path.join(TILES_DIR, "*.jpg")) + glob(os.path.join(TILES_DIR, "*.png"))
print(f"Найдено {len(tile_files)} тайлов для обработки.")

image_data = {}

with torch.no_grad():
    for tile_path in tqdm(tile_files, desc="Инференс тайлов"):
        filename  = os.path.basename(tile_path)
        parts     = os.path.splitext(filename)[0].rsplit('_', 2)
        base_name = parts[0]

        if base_name not in image_data:
            image_data[base_name] = []

        image  = cv2.imread(tile_path)
        image  = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)

        upsampled_logits = torch.nn.functional.interpolate(
            outputs.logits,
            size=image.shape[:2],
            mode="bilinear",
            align_corners=False
        )

        # Softmax-вероятности вместо argmax — ловим слабые повреждения
        probs = torch.nn.functional.softmax(upsampled_logits, dim=1).squeeze(0).cpu().numpy()

        tile_ratios = {}
        for class_idx, class_name in CLASS_MAPPING.items():
            if class_name == "background":
                continue
            class_probs = probs[class_idx]  # (H, W)
            tile_ratios[class_name] = float((class_probs > CONFIDENCE_THRESHOLD).mean())

        image_data[base_name].append(tile_ratios)  


Найдено 6693 тайлов для обработки.


Инференс тайлов:   0%|          | 0/6693 [00:00<?, ?it/s]

## 4. Агрегация признаков и построение 15D векторов

Считаем: mean, max, prevalence.

In [6]:
from scipy.stats import skew as scipy_skew

results = []

for base_name, tiles in tqdm(image_data.items(), desc="Агрегация признаков"):
    num_tiles = len(tiles)

    # Диагностика — убери после исправления
    if num_tiles == 0:
        print(f"  [SKIP] '{base_name}' — 0 тайлов в image_data")
        continue
        
    vector_dict = {"building_name": base_name, "num_tiles": num_tiles}

    # Собираем все tile-ratios по каждому классу в массив
    class_ratios = {
        cls: np.array([t.get(cls, 0.0) for t in tiles])
        for cls in CLASSES_TO_TRACK
    }

    for cls in CLASSES_TO_TRACK:
        ratios = class_ratios[cls]  # numpy array длиной = num_tiles

        # --- Исходные 3 признака (остаются) ---
        mean_val   = float(np.mean(ratios))
        max_val    = float(np.max(ratios))
        prevalence = float(np.mean(ratios > EPSILON))

        # --- Новые статистические признаки по тайлам ---
        std_val = float(np.std(ratios))
        q75_val = float(np.percentile(ratios, 75))

        # skewness: > 0 → редкие тяжёлые тайлы (локальная катастрофа)
        #           ≈ 0 → дефект распределён равномерно
        # При num_tiles < 3 нет смысла считать — ставим 0
        skew_val = float(scipy_skew(ratios)) if num_tiles >= 3 else 0.0

        # --- Производные признаки (нелинейность) ---
        # Концентрация: насколько max "выбивается" над средним
        # Высокое значение → одна локальная точка катастрофы при чистом фасаде
        concentration = max_val / (mean_val + 1e-6)
        concentration = min(concentration, 50.0)  # clip от экстремальных делений

        # Совместная оценка: серьёзность × охват
        severity_coverage = max_val * prevalence

        vector_dict[f"{cls}_mean"]             = round(mean_val, 4)
        vector_dict[f"{cls}_max"]              = round(max_val, 4)
        vector_dict[f"{cls}_prevalence"]       = round(prevalence, 4)
        vector_dict[f"{cls}_std"]              = round(std_val, 4)
        vector_dict[f"{cls}_q75"]              = round(q75_val, 4)
        vector_dict[f"{cls}_skewness"]         = round(skew_val, 4)
        vector_dict[f"{cls}_concentration"]    = round(concentration, 4)
        vector_dict[f"{cls}_severity_coverage"]= round(severity_coverage, 4)

    results.append(vector_dict)

# Итоговый датафрейм
df = pd.DataFrame(results)

# Порядок колонок
AGGREGATIONS = ["mean", "max", "prevalence", "std", "q75", "skewness",
                "concentration", "severity_coverage"]
columns_order = ["building_name", "num_tiles"] + [
    f"{cls}_{agg}" for cls in CLASSES_TO_TRACK for agg in AGGREGATIONS
]
df = df[columns_order]

# Сохраняем
CSV_OUTPUT = "/kaggle/working/building_features_40d.csv"
df.to_csv(CSV_OUTPUT, index=False)
print(f"Извлечено {len(df)} зданий.")
print(f"Размерность вектора: {len(AGGREGATIONS) * len(CLASSES_TO_TRACK)}D  "
      f"(было 15D, стало {len(AGGREGATIONS) * len(CLASSES_TO_TRACK)}D)")
print(f"Сохранено: {CSV_OUTPUT}")
display(df.head())


Агрегация признаков:   0%|          | 0/300 [00:00<?, ?it/s]

Извлечено 300 зданий.
Размерность вектора: 40D  (было 15D, стало 40D)
Сохранено: /kaggle/working/building_features_40d.csv


,building_name,num_tiles,coating_deterioration_mean,coating_deterioration_max,coating_deterioration_prevalence,coating_deterioration_std,coating_deterioration_q75,coating_deterioration_skewness,coating_deterioration_concentration,coating_deterioration_severity_coverage,...,moisture_bio_damage_concentration,moisture_bio_damage_severity_coverage,vandalism_mean,vandalism_max,vandalism_prevalence,vandalism_std,vandalism_q75,vandalism_skewness,vandalism_concentration,vandalism_severity_coverage
0,DSC02518,24,0.0063,0.0357,0.0000,0.0092,0.0086,1.8683,5.6975,0.0000,...,5.5882,0.0857,0.0011,0.0133,0.0,0.0029,0.0001,3.3036,12.4639,0.0
1,2026-04-09 17-36-25,12,0.0161,0.0548,0.1667,0.0187,0.0226,1.0639,3.4077,0.0091,...,5.2847,0.0047,0.0001,0.0004,0.0,0.0001,0.0000,2.0781,7.4315,0.0
2,IMG_20260410_142531328_HDR 2,12,0.0347,0.1230,0.4167,0.0375,0.0570,0.9944,3.5482,0.0513,...,4.4841,0.0057,0.0034,0.0185,0.0,0.0064,0.0019,1.6938,5.4341,0.0
3,62,12,0.0790,0.2156,0.5000,0.0775,0.1309,0.4853,2.7305,0.1078,...,8.0153,0.0000,0.0023,0.0234,0.0,0.0065,0.0000,2.8874,10.3247,0.0
4,DSC02333,24,0.0021,0.0211,0.0000,0.0047,0.0011,2.9449,9.9065,0.0000,...,8.2543,0.0120,0.0000,0.0000,0.0,0.0000,0.0000,NaN,0.0000,0.0
